# Threshold Sensitivity Analysis and Cost-Benefit Simulation

## Learning Objectives

In this notebook, you will learn:
- How to perform threshold sensitivity analysis for drift detection
- How to simulate the cost-benefit trade-off of different thresholds
- How to calibrate thresholds based on business requirements
- How to design multi-level alert systems
- How to balance false positives and false negatives

## Introduction

Setting the right threshold for drift detection is a critical decision that involves balancing:

- **False Positives (False Alarms)**: Detecting drift when none exists
  - Cost: Unnecessary model retraining, wasted resources, alert fatigue

- **False Negatives (Missed Drift)**: Failing to detect actual drift
  - Cost: Degraded model performance, poor predictions, business impact

The optimal threshold depends on the specific business context and the relative costs of these two types of errors.

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import wasserstein_distance

# Set random seed for reproducibility
np.random.seed(42)

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

## 1. Generate Synthetic Data with Known Drift

In [ ]:
def generate_monitoring_data(n_periods=100, drift_periods=None, drift_magnitude=0.5):
    """
    Generate synthetic monitoring data with known drift periods.
    
    Parameters:
    -----------
    n_periods : int
        Number of monitoring periods
    drift_periods : list
        List of period indices where drift occurs
    drift_magnitude : float
        Magnitude of drift (shift in mean)
    
    Returns:
    --------
    data : dict
        Dictionary containing reference data, monitored data, and ground truth
    """
    if drift_periods is None:
        drift_periods = [30, 60, 80]  # Default drift periods
    
    reference_data = np.random.normal(0, 1, 1000)
    monitored_data_list = []
    ground_truth = []  # True drift labels
    
    for period in range(n_periods):
        if period in drift_periods:
            # Drift present
            data = np.random.normal(drift_magnitude, 1, 1000)
            ground_truth.append(1)
        else:
            # No drift
            data = np.random.normal(0, 1, 1000)
            ground_truth.append(0)
        
        monitored_data_list.append(data)
    
    return {
        'reference': reference_data,
        'monitored': monitored_data_list,
        'ground_truth': np.array(ground_truth),
        'drift_periods': drift_periods
    }

# Generate data
data = generate_monitoring_data(n_periods=100, drift_periods=[30, 60, 80], drift_magnitude=0.5)

print(f"Generated {len(data['monitored'])} monitoring periods")
print(f"Drift occurs at periods: {data['drift_periods']}")
print(f"Total drift periods: {sum(data['ground_truth'])}")

## 2. Calculate Drift Metrics for All Periods

In [ ]:
def calculate_psi(reference, monitored, bins=10):
    """Calculate Population Stability Index."""
    bin_edges = np.percentile(reference, np.linspace(0, 100, bins + 1))
    bin_edges = np.unique(bin_edges)
    
    ref_counts, _ = np.histogram(reference, bins=bin_edges)
    mon_counts, _ = np.histogram(monitored, bins=bin_edges)
    
    ref_percents = ref_counts / len(reference)
    mon_percents = mon_counts / len(monitored)
    
    epsilon = 1e-10
    ref_percents = np.where(ref_percents == 0, epsilon, ref_percents)
    mon_percents = np.where(mon_percents == 0, epsilon, mon_percents)
    
    psi = np.sum((mon_percents - ref_percents) * np.log(mon_percents / ref_percents))
    return psi

# Calculate metrics for all periods
psi_values = []
ks_statistics = []
ks_pvalues = []
wasserstein_distances = []

for monitored in data['monitored']:
    # PSI
    psi = calculate_psi(data['reference'], monitored)
    psi_values.append(psi)
    
    # KS test
    ks_stat, ks_pval = stats.ks_2samp(data['reference'], monitored)
    ks_statistics.append(ks_stat)
    ks_pvalues.append(ks_pval)
    
    # Wasserstein distance
    wass = wasserstein_distance(data['reference'], monitored)
    wasserstein_distances.append(wass)

# Create DataFrame
metrics_df = pd.DataFrame({
    'Period': range(len(data['monitored'])),
    'PSI': psi_values,
    'KS_Statistic': ks_statistics,
    'KS_PValue': ks_pvalues,
    'Wasserstein': wasserstein_distances,
    'True_Drift': data['ground_truth']
})

print("Drift Metrics Summary:")
print("=" * 60)
print(metrics_df.describe())
print("\nDrift periods metrics:")
print(metrics_df[metrics_df['True_Drift'] == 1][['Period', 'PSI', 'KS_Statistic', 'Wasserstein']])

## 3. Visualize Metrics Over Time

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 12))

# PSI over time
axes[0].plot(metrics_df['Period'], metrics_df['PSI'], linewidth=2, label='PSI')
axes[0].axhline(y=0.1, color='orange', linestyle='--', label='Moderate Threshold (0.1)')
axes[0].axhline(y=0.2, color='red', linestyle='--', label='Significant Threshold (0.2)')
for dp in data['drift_periods']:
    axes[0].axvline(x=dp, color='gray', linestyle=':', alpha=0.5)
axes[0].fill_between(metrics_df['Period'], 0, metrics_df['PSI'], 
                     where=metrics_df['True_Drift']==1, alpha=0.3, color='red', label='True Drift')
axes[0].set_ylabel('PSI Value')
axes[0].set_title('Population Stability Index Over Time', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# KS statistic over time
axes[1].plot(metrics_df['Period'], metrics_df['KS_Statistic'], linewidth=2, color='green', label='KS Statistic')
for dp in data['drift_periods']:
    axes[1].axvline(x=dp, color='gray', linestyle=':', alpha=0.5)
axes[1].fill_between(metrics_df['Period'], 0, metrics_df['KS_Statistic'], 
                     where=metrics_df['True_Drift']==1, alpha=0.3, color='red', label='True Drift')
axes[1].set_ylabel('KS Statistic')
axes[1].set_title('Kolmogorov-Smirnov Statistic Over Time', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Wasserstein distance over time
axes[2].plot(metrics_df['Period'], metrics_df['Wasserstein'], linewidth=2, color='purple', label='Wasserstein Distance')
for dp in data['drift_periods']:
    axes[2].axvline(x=dp, color='gray', linestyle=':', alpha=0.5)
axes[2].fill_between(metrics_df['Period'], 0, metrics_df['Wasserstein'], 
                     where=metrics_df['True_Drift']==1, alpha=0.3, color='red', label='True Drift')
axes[2].set_xlabel('Monitoring Period')
axes[2].set_ylabel('Wasserstein Distance')
axes[2].set_title('Wasserstein Distance Over Time', fontsize=14, fontweight='bold')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Threshold Sensitivity Analysis

In [ ]:
def evaluate_threshold(metric_values, ground_truth, threshold):
    """
    Evaluate a threshold for drift detection.
    
    Returns:
    --------
    metrics : dict
        Dictionary containing TP, FP, TN, FN, precision, recall, F1
    """
    predictions = (metric_values >= threshold).astype(int)
    
    TP = np.sum((predictions == 1) & (ground_truth == 1))  # True Positives
    FP = np.sum((predictions == 1) & (ground_truth == 0))  # False Positives
    TN = np.sum((predictions == 0) & (ground_truth == 0))  # True Negatives
    FN = np.sum((predictions == 0) & (ground_truth == 1))  # False Negatives
    
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    return {
        'TP': TP, 'FP': FP, 'TN': TN, 'FN': FN,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

# Test different PSI thresholds
psi_thresholds = np.linspace(0.01, 0.5, 50)
psi_results = []

for threshold in psi_thresholds:
    result = evaluate_threshold(metrics_df['PSI'].values, metrics_df['True_Drift'].values, threshold)
    result['threshold'] = threshold
    psi_results.append(result)

psi_results_df = pd.DataFrame(psi_results)

# Plot precision-recall curve
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Precision-Recall curve
axes[0, 0].plot(psi_results_df['recall'], psi_results_df['precision'], linewidth=2, marker='o', markersize=3)
axes[0, 0].set_xlabel('Recall (True Positive Rate)')
axes[0, 0].set_ylabel('Precision')
axes[0, 0].set_title('Precision-Recall Curve (PSI)', fontsize=14, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

# F1 score vs threshold
axes[0, 1].plot(psi_results_df['threshold'], psi_results_df['f1'], linewidth=2, color='green')
best_f1_idx = psi_results_df['f1'].idxmax()
best_threshold = psi_results_df.loc[best_f1_idx, 'threshold']
axes[0, 1].axvline(x=best_threshold, color='red', linestyle='--', 
                  label=f'Best F1 Threshold: {best_threshold:.3f}')
axes[0, 1].axvline(x=0.1, color='orange', linestyle='--', label='Standard Threshold: 0.1')
axes[0, 1].axvline(x=0.2, color='purple', linestyle='--', label='Standard Threshold: 0.2')
axes[0, 1].set_xlabel('PSI Threshold')
axes[0, 1].set_ylabel('F1 Score')
axes[0, 1].set_title('F1 Score vs Threshold', fontsize=14, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# False Positives vs False Negatives
axes[1, 0].plot(psi_results_df['threshold'], psi_results_df['FP'], linewidth=2, label='False Positives', color='red')
axes[1, 0].plot(psi_results_df['threshold'], psi_results_df['FN'], linewidth=2, label='False Negatives', color='blue')
axes[1, 0].axvline(x=best_threshold, color='gray', linestyle='--', alpha=0.5)
axes[1, 0].set_xlabel('PSI Threshold')
axes[1, 0].set_ylabel('Count')
axes[1, 0].set_title('False Positives vs False Negatives', fontsize=14, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Precision and Recall vs Threshold
axes[1, 1].plot(psi_results_df['threshold'], psi_results_df['precision'], linewidth=2, label='Precision', color='green')
axes[1, 1].plot(psi_results_df['threshold'], psi_results_df['recall'], linewidth=2, label='Recall', color='orange')
axes[1, 1].axvline(x=best_threshold, color='red', linestyle='--', alpha=0.5, label=f'Best F1: {best_threshold:.3f}')
axes[1, 1].set_xlabel('PSI Threshold')
axes[1, 1].set_ylabel('Score')
axes[1, 1].set_title('Precision and Recall vs Threshold', fontsize=14, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nBest F1 Score: {psi_results_df.loc[best_f1_idx, 'f1']:.3f}")
print(f"Best Threshold: {best_threshold:.3f}")
print(f"At this threshold:")
print(f"  Precision: {psi_results_df.loc[best_f1_idx, 'precision']:.3f}")
print(f"  Recall: {psi_results_df.loc[best_f1_idx, 'recall']:.3f}")
print(f"  False Positives: {psi_results_df.loc[best_f1_idx, 'FP']:.0f}")
print(f"  False Negatives: {psi_results_df.loc[best_f1_idx, 'FN']:.0f}")

## 5. Cost-Benefit Simulation

In [ ]:
def calculate_total_cost(predictions, ground_truth, cost_fp, cost_fn):
    """
    Calculate total cost based on false positives and false negatives.
    
    Parameters:
    -----------
    predictions : array
        Binary predictions (1 = drift detected, 0 = no drift)
    ground_truth : array
        True drift labels
    cost_fp : float
        Cost of a false positive (unnecessary retraining)
    cost_fn : float
        Cost of a false negative (missed drift, model degradation)
    
    Returns:
    --------
    total_cost : float
        Total cost
    """
    FP = np.sum((predictions == 1) & (ground_truth == 0))
    FN = np.sum((predictions == 0) & (ground_truth == 1))
    
    total_cost = (FP * cost_fp) + (FN * cost_fn)
    return total_cost

# Define cost scenarios
cost_scenarios = {
    'Healthcare (High FN Cost)': {'cost_fp': 1000, 'cost_fn': 100000},
    'Finance (Moderate Costs)': {'cost_fp': 5000, 'cost_fn': 20000},
    'E-commerce (Low FN Cost)': {'cost_fp': 2000, 'cost_fn': 5000}
}

# Calculate costs for each scenario
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (scenario_name, costs) in enumerate(cost_scenarios.items()):
    scenario_costs = []
    
    for threshold in psi_thresholds:
        predictions = (metrics_df['PSI'].values >= threshold).astype(int)
        total_cost = calculate_total_cost(predictions, metrics_df['True_Drift'].values, 
                                          costs['cost_fp'], costs['cost_fn'])
        scenario_costs.append(total_cost)
    
    # Find optimal threshold
    optimal_idx = np.argmin(scenario_costs)
    optimal_threshold = psi_thresholds[optimal_idx]
    optimal_cost = scenario_costs[optimal_idx]
    
    # Plot
    axes[idx].plot(psi_thresholds, scenario_costs, linewidth=2)
    axes[idx].axvline(x=optimal_threshold, color='red', linestyle='--', 
                     label=f'Optimal: {optimal_threshold:.3f}')
    axes[idx].axvline(x=0.1, color='orange', linestyle=':', label='Standard: 0.1')
    axes[idx].axvline(x=0.2, color='purple', linestyle=':', label='Standard: 0.2')
    axes[idx].scatter([optimal_threshold], [optimal_cost], color='red', s=100, zorder=5)
    axes[idx].set_xlabel('PSI Threshold')
    axes[idx].set_ylabel('Total Cost ($)')
    axes[idx].set_title(f'{scenario_name}\nFP Cost: ${costs["cost_fp"]}, FN Cost: ${costs["cost_fn"]}', 
                       fontsize=12, fontweight='bold')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)
    
    print(f"\n{scenario_name}:")
    print(f"  Optimal Threshold: {optimal_threshold:.3f}")
    print(f"  Minimum Cost: ${optimal_cost:,.0f}")
    
    # Compare with standard thresholds
    pred_01 = (metrics_df['PSI'].values >= 0.1).astype(int)
    cost_01 = calculate_total_cost(pred_01, metrics_df['True_Drift'].values, 
                                   costs['cost_fp'], costs['cost_fn'])
    pred_02 = (metrics_df['PSI'].values >= 0.2).astype(int)
    cost_02 = calculate_total_cost(pred_02, metrics_df['True_Drift'].values, 
                                   costs['cost_fp'], costs['cost_fn'])
    
    print(f"  Cost at 0.1 threshold: ${cost_01:,.0f} ({((cost_01-optimal_cost)/optimal_cost*100):.1f}% higher)")
    print(f"  Cost at 0.2 threshold: ${cost_02:,.0f} ({((cost_02-optimal_cost)/optimal_cost*100):.1f}% higher)")

plt.tight_layout()
plt.show()

## 6. Multi-Level Alert System Design

In [ ]:
def design_alert_system(metric_values, warning_threshold, action_threshold):
    """
    Design a multi-level alert system.
    
    Returns:
    --------
    alerts : array
        Alert levels (0 = no alert, 1 = warning, 2 = action)
    """
    alerts = np.zeros(len(metric_values))
    alerts[metric_values >= warning_threshold] = 1  # Warning
    alerts[metric_values >= action_threshold] = 2   # Action
    return alerts

# Design alert system with two thresholds
warning_threshold = 0.08
action_threshold = 0.15

alerts = design_alert_system(metrics_df['PSI'].values, warning_threshold, action_threshold)

# Visualize alert system
fig, ax = plt.subplots(figsize=(16, 6))

# Plot PSI values
ax.plot(metrics_df['Period'], metrics_df['PSI'], linewidth=2, label='PSI', color='blue')

# Add threshold lines
ax.axhline(y=warning_threshold, color='orange', linestyle='--', linewidth=2, label=f'Warning Threshold ({warning_threshold})')
ax.axhline(y=action_threshold, color='red', linestyle='--', linewidth=2, label=f'Action Threshold ({action_threshold})')

# Highlight alert zones
ax.fill_between(metrics_df['Period'], 0, metrics_df['PSI'], 
               where=(alerts == 1), alpha=0.2, color='orange', label='Warning Zone')
ax.fill_between(metrics_df['Period'], 0, metrics_df['PSI'], 
               where=(alerts == 2), alpha=0.3, color='red', label='Action Zone')

# Mark true drift periods
for dp in data['drift_periods']:
    ax.axvline(x=dp, color='gray', linestyle=':', alpha=0.5)

ax.set_xlabel('Monitoring Period', fontsize=12)
ax.set_ylabel('PSI Value', fontsize=12)
ax.set_title('Multi-Level Alert System', fontsize=14, fontweight='bold')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Summary statistics
print("\nAlert System Summary:")
print("=" * 60)
print(f"Total periods: {len(alerts)}")
print(f"No alert (green): {np.sum(alerts == 0)} periods")
print(f"Warning (orange): {np.sum(alerts == 1)} periods")
print(f"Action (red): {np.sum(alerts == 2)} periods")
print(f"\nTrue drift periods: {sum(data['ground_truth'])}")
print(f"Action alerts during drift: {np.sum((alerts == 2) & (metrics_df['True_Drift'] == 1))}")
print(f"Warning alerts during drift: {np.sum((alerts == 1) & (metrics_df['True_Drift'] == 1))}")

## 7. Practical Recommendation Engine

In [ ]:
def recommend_threshold(cost_fp, cost_fn, metric='PSI'):
    """
    Recommend optimal threshold based on cost parameters.
    """
    thresholds = np.linspace(0.01, 0.5, 100)
    costs = []
    
    for threshold in thresholds:
        predictions = (metrics_df[metric].values >= threshold).astype(int)
        total_cost = calculate_total_cost(predictions, metrics_df['True_Drift'].values, 
                                          cost_fp, cost_fn)
        costs.append(total_cost)
    
    optimal_idx = np.argmin(costs)
    optimal_threshold = thresholds[optimal_idx]
    
    # Calculate performance metrics
    predictions = (metrics_df[metric].values >= optimal_threshold).astype(int)
    result = evaluate_threshold(metrics_df[metric].values, metrics_df['True_Drift'].values, 
                               optimal_threshold)
    
    return {
        'threshold': optimal_threshold,
        'total_cost': costs[optimal_idx],
        'precision': result['precision'],
        'recall': result['recall'],
        'f1': result['f1'],
        'fp_count': result['FP'],
        'fn_count': result['FN']
    }

# Interactive recommendation
print("Threshold Recommendation Engine")
print("=" * 60)
print("\nEnter your cost parameters:")
print("\nExample scenarios:")
print("  Healthcare: FP=$1,000, FN=$100,000")
print("  Finance: FP=$5,000, FN=$20,000")
print("  E-commerce: FP=$2,000, FN=$5,000")

# Use default values for demonstration
cost_fp = 5000
cost_fn = 20000

recommendation = recommend_threshold(cost_fp, cost_fn)

print(f"\nRecommended Configuration:")
print("=" * 60)
print(f"Optimal PSI Threshold: {recommendation['threshold']:.3f}")
print(f"\nExpected Performance:")
print(f"  Precision: {recommendation['precision']:.2%}")
print(f"  Recall: {recommendation['recall']:.2%}")
print(f"  F1 Score: {recommendation['f1']:.2%}")
print(f"\nExpected Costs (per 100 periods):")
print(f"  False Positives: {recommendation['fp_count']:.0f} × ${cost_fp:,} = ${recommendation['fp_count'] * cost_fp:,.0f}")
print(f"  False Negatives: {recommendation['fn_count']:.0f} × ${cost_fn:,} = ${recommendation['fn_count'] * cost_fn:,.0f}")
print(f"  Total Cost: ${recommendation['total_cost']:,.0f}")
print(f"\nRecommendation:")
if recommendation['threshold'] < 0.1:
    print("  → Use a more sensitive threshold than the standard 0.1")
    print("  → This is appropriate for high-cost-of-failure scenarios")
elif recommendation['threshold'] > 0.2:
    print("  → Use a less sensitive threshold than the standard 0.2")
    print("  → This is appropriate when false alarms are costly")
else:
    print("  → Standard thresholds (0.1-0.2) are appropriate")
    print("  → Consider using a two-tier alert system")

## Key Takeaways

### Threshold Selection Principles

1. **No one-size-fits-all**: Optimal thresholds depend on business context
2. **Cost-benefit analysis**: Balance false positives vs false negatives
3. **Domain-specific**: Healthcare requires different thresholds than e-commerce
4. **Multi-level alerts**: Use warning and action zones for better control
5. **Continuous calibration**: Update thresholds based on operational feedback

### Cost Considerations

| Domain | FP Cost | FN Cost | Recommended Approach |
|--------|---------|---------|---------------------|
| Healthcare | Low | Very High | Low threshold, high sensitivity |
| Finance | Moderate | High | Balanced threshold |
| E-commerce | Moderate | Moderate | Standard thresholds |
| Consumer Web | High | Low | High threshold, low sensitivity |

### Best Practices

1. **Start with standard thresholds** (PSI: 0.1-0.2) and adjust based on experience
2. **Use multi-level alerts** to provide graduated responses
3. **Monitor false positive/negative rates** and adjust thresholds accordingly
4. **Document threshold decisions** and rationale for future reference
5. **Combine multiple metrics** for more robust drift detection
6. **A/B test thresholds** in production to validate performance
7. **Review and recalibrate** thresholds periodically

### Alert System Design

**Two-Tier System:**
- **Warning Zone**: Monitor closely, investigate root cause
- **Action Zone**: Trigger automated retraining or manual intervention

**Three-Tier System:**
- **Green (No Alert)**: Normal operation
- **Yellow (Warning)**: Increased monitoring
- **Red (Action)**: Immediate response required

## Exercises

1. Implement a dynamic threshold that adapts based on recent false positive/negative rates.

2. Create a cost calculator that helps stakeholders determine their FP and FN costs.

3. Design an alert fatigue prevention system that adjusts thresholds when too many alerts are triggered.

4. Build a dashboard that visualizes the cost-benefit trade-off for different threshold settings.